# Medium World — Mixed → Pair controls (five paired seeds)

Run seeds **42–46** across four matched-exposure curricula using the existing Medium World model and dataset. The notebook saves each completed run and its real prediction trace, and safely skips verified completed runs on rerun. No pilot/full mode switches are needed.


In [1]:
from pathlib import Path
import sys
PROJECT_ROOT = Path().cwd().parent.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

SEEDS = [42, 43, 44, 45, 46]
MAIN_EPOCHS, COMMON_EPOCHS = 36, 9
BATCH_SIZE, EVAL_EVERY = 50, 100
LR, WEIGHT_DECAY = 1e-4, 0.0
MODEL_DIM, PATCH_SIZE, MODEL_DEPTH = 48, 4, 1
AUGMENT, DEVICE = True, "cuda"
SAVE_CHECKPOINTS = False

DATA_REL = Path("outputs/medium_world/dataset")
OUT_REL = Path("outputs/medium_world/experiments/mixed_pair_control/five_seeds")
DATA_DIR = PROJECT_ROOT / DATA_REL
OUT_DIR = PROJECT_ROOT / OUT_REL
RESULTS_FILE = OUT_DIR / "schedule_comparison.json"
TRACE_DIR = OUT_DIR / "prediction_traces"
CHECKPOINT_DIR = OUT_DIR / "checkpoints"

In [2]:
import json, hashlib
from collections import Counter
import numpy as np
import pandas as pd
import torch
from torch import nn
from IPython.display import display
from src.medium_visual_world.learning import (
    TASKS, MediumDataset, AttributeBindingViT, augment_images, set_seed, environment
)
from src.medium_visual_world.order_control import SCHEDULES, make_order_control_blocks
from src.medium_visual_world.prediction_tracking import (
    PredictionTrace, evaluate_predictions, load_trace
)

device = torch.device(DEVICE)
if device.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("CUDA was requested but is unavailable")
torch.set_num_threads(min(4, torch.get_num_threads()))
train_data = MediumDataset(DATA_DIR / "train.jsonl")
id_dev = MediumDataset(DATA_DIR / "id_dev.jsonl")
ood_dev = MediumDataset(DATA_DIR / "ood_dev.jsonl")
hashes = {name: hashlib.sha256((DATA_DIR / name).read_bytes()).hexdigest()
          for name in ("train.jsonl", "id_dev.jsonl", "ood_dev.jsonl")}
MAIN_STEPS = len(train_data) * MAIN_EPOCHS // BATCH_SIZE
TOTAL_STEPS = len(train_data) * (MAIN_EPOCHS + COMMON_EPOCHS) // BATCH_SIZE
assert MAIN_STEPS == 7200 and TOTAL_STEPS == 9000
assert all(train_data.tasks.count(task) == 2500 for task in TASKS)
print("Device:", device, "| Steps/run:", TOTAL_STEPS,
      "| Seeds:", SEEDS, "| Schedules:", len(SCHEDULES))

Device: cuda | Steps/run: 9000 | Seeds: [42, 43, 44, 45, 46] | Schedules: 4


## Verify the exact exposure schedules

Check the real dataset, not a toy substitute. These assertions prevent starting an expensive run with a different protocol.

In [3]:
def check_plan(blocks, plan, schedule, seed):
    assert sum(len(ids) for _, ids in blocks) == len(train_data) * 45
    assert all(len(ids) % BATCH_SIZE == 0 for _, ids in blocks)
    assert sum(len(ids) for _, ids in blocks) // BATCH_SIZE == TOTAL_STEPS
    assert all(n == 112500 for n in plan["counts_by_template"].values())
    assert {int(n) for n in Counter(i for _, ids in blocks for i in ids).values()} == {45}
    assert len(set(i for _, ids in blocks for i in ids)) == len(train_data)
    return {"schedule": schedule, "seed": seed, "phases": " → ".join(
        f"{name} ({len(ids)//BATCH_SIZE})" for name, ids in blocks)}

rows = []
for schedule in SCHEDULES:
    blocks, plan = make_order_control_blocks(train_data.rows, schedule, 42,
                                            MAIN_EPOCHS, COMMON_EPOCHS)
    rows.append(check_plan(blocks, plan, schedule, 42))
    if schedule == SCHEDULES[0]:
        reference_pair = blocks[-2][1]
        reference_common = blocks[-1][1]
    assert blocks[-1][1] == reference_common, "Common consolidation differs"
    if schedule in SCHEDULES[:3]:
        assert blocks[-2][0] == "pair_focus"
        assert blocks[-2][1] == reference_pair, "Pair-focus sequence differs"
display(pd.DataFrame(rows))
print("PASS: 9,000 updates, 45 presentations/question and matched final blocks")

,schedule,seed,phases
0,staged_pair_focus,42,attributes (3600) → conjunction (1800) → pair_...
1,prereq_mixed_pair_focus,42,prereq_mixed (5400) → pair_early (900) → pair_...
2,mixed_pair_focus,42,mixed_prefix (6300) → pair_focus (900) → commo...
3,mixed,42,mixed (7200) → common_mixed (1800)


PASS: 9,000 updates, 45 presentations/question and matched final blocks


## Train all five seeds

Every run uses the same 9,000 updates, learning rate, and task exposure. The notebook evaluates every 100 updates, saves a compressed prediction trace and aggregate results after each completed run, and skips runs only when their saved trace passes validation. A stopped run can be resumed by rerunning the notebook.


In [4]:
def binding_auc(history):
    pairs = [(h["step"], h["id_dev"]["by_template"]["pair"]) for h in history]
    assert pairs[0][0] == 0 and pairs[-1][0] == TOTAL_STEPS
    return sum((b[0]-a[0])*(a[1]+b[1])/2
               for a,b in zip(pairs[:-1],pairs[1:])) / TOTAL_STEPS

def validate_trace(path, seed, schedule, strict=True):
    trace = load_trace(path)
    info = trace["metadata"]
    assert int(info["seed"]) == seed and info["schedule"] == schedule
    for split, dataset in (("id_dev", id_dev), ("ood_dev", ood_dev)):
        t = trace["splits"][split]
        assert len(t["ids"]) == len(dataset)
        assert list(t["ids"]) == [str(r["id"]) for r in dataset.rows]
        assert np.array_equal(t["steps"], np.arange(0, TOTAL_STEPS+1, EVAL_EVERY))
        assert t["predictions"].shape == (TOTAL_STEPS//EVAL_EVERY+1, len(dataset))
        assert np.isfinite(t["probabilities"]).all()
    return trace

def run_one(schedule, seed):
    set_seed(seed)
    model = AttributeBindingViT(dim=MODEL_DIM, patch_size=PATCH_SIZE,
                                depth=MODEL_DEPTH).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
    blocks, plan = make_order_control_blocks(train_data.rows, schedule, seed,
                                             MAIN_EPOCHS, COMMON_EPOCHS)
    check_plan(blocks, plan, schedule, seed)
    history, step = [], 0
    accumulated, batches = 0.0, 0
    seen_tasks = Counter()
    trace = PredictionTrace({"id_dev": id_dev, "ood_dev": ood_dev},
                            metadata={"seed": seed, "schedule": schedule,
                                      "dataset_hashes": hashes})
    transitions = []

    def evaluate_step(phase):
        nonlocal accumulated, batches
        scores = {}
        for split, data in (("id_dev", id_dev), ("ood_dev", ood_dev)):
            result, predictions = evaluate_predictions(model, data, device, BATCH_SIZE)
            scores[split] = result
            trace.add(split, step, predictions)
        record = {"step": step, "phase": phase, "lr": optimizer.param_groups[0]["lr"],
                  "batch_train_loss": accumulated/batches if batches else None,
                  "cumulative_task_presentations": dict(seen_tasks), **scores}
        history.append(record)
        accumulated, batches = 0.0, 0
        print(f"{schedule:24} seed={seed} step={step:4} "
              f"ID pair={scores['id_dev']['by_template']['pair']:.3f} "
              f"OOD pair={scores['ood_dev']['by_template']['pair']:.3f}", flush=True)

    evaluate_step("initial")
    for phase, ids in blocks:
        start_step = step
        for start in range(0, len(ids), BATCH_SIZE):
            batch_ids = ids[start:start+BATCH_SIZE]
            x, q, y = train_data.batch(batch_ids, device)
            if AUGMENT:
                x = augment_images(x)
            model.train()
            optimizer.zero_grad(set_to_none=True)
            loss = nn.functional.cross_entropy(model(x, q), y)
            if not torch.isfinite(loss):
                raise FloatingPointError(f"Nonfinite loss: {schedule} seed={seed} step={step}")
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            step += 1
            accumulated += loss.item()
            batches += 1
            seen_tasks.update(train_data.tasks[i] for i in batch_ids)
            if step % EVAL_EVERY == 0:
                evaluate_step(phase)
        assert history[-1]["step"] == step, "Every phase boundary must be evaluated"
        transitions.append({"phase": phase, "start_step": start_step,
                            "end_step": step, "reason": "fixed_schedule"})
        if SAVE_CHECKPOINTS:
            CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
            torch.save({"model": model.state_dict(), "seed": seed,
                        "schedule": schedule, "step": step},
                       CHECKPOINT_DIR / f"{schedule}_seed_{seed}_step_{step}.pt")
    assert step == TOTAL_STEPS and len(history) == TOTAL_STEPS//EVAL_EVERY + 1
    assert all(seen_tasks[task] == 112500 for task in TASKS)
    TRACE_DIR.mkdir(parents=True, exist_ok=True)
    trace_rel = Path("prediction_traces") / f"{schedule}_seed_{seed}.npz"
    trace.save(OUT_DIR / trace_rel)
    validate_trace(OUT_DIR / trace_rel, seed, schedule)
    return {"seed": seed, "schedule": schedule, "status": "completed",
            "steps_run": step, "phase_plan": plan, "transitions": transitions,
            "history": history, "final": history[-1],
            "id_pair_aulc": binding_auc(history),
            "prediction_trace": trace_rel.as_posix()}

In [5]:
CONFIG = {"schedules": list(SCHEDULES), "main_epochs": MAIN_EPOCHS,
          "common_epochs": COMMON_EPOCHS, "batch_size": BATCH_SIZE,
          "eval_every": EVAL_EVERY, "learning_rate": LR,
          "weight_decay": WEIGHT_DECAY, "dim": MODEL_DIM,
          "patch_size": PATCH_SIZE, "depth": MODEL_DEPTH,
          "augmentation": AUGMENT, "total_steps": TOTAL_STEPS,
          "trace_schema": 1}

def read_existing(path):
    report = json.loads(path.read_text(encoding="utf-8"))
    if report["config"] != CONFIG or report["dataset_hashes"] != hashes:
        raise ValueError(f"Output configuration or dataset changed: {path}")
    return report

def write_report(report, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".json.tmp")
    tmp.write_text(json.dumps(report, indent=2)+"\n", encoding="utf-8")
    tmp.replace(path)

if RESULTS_FILE.exists():
    report = read_existing(RESULTS_FILE)
else:
    report = {"status": "exploratory", "config": CONFIG,
              "dataset_hashes": hashes, "environment": environment(), "runs": []}

completed = {(r["seed"], r["schedule"]) for r in report["runs"]
             if r["status"] == "completed"}
for seed in SEEDS:
    for schedule in SCHEDULES:
        if (seed, schedule) in completed:
            matching = next(r for r in report["runs"] if r["seed"] == seed
                            and r["schedule"] == schedule)
            validate_trace(OUT_DIR / matching["prediction_trace"], seed, schedule)
            print("Skipping verified run:", seed, schedule)
            continue
        result = run_one(schedule, seed)
        report["runs"].append(result)
        write_report(report, RESULTS_FILE)
        print("Saved:", seed, schedule, RESULTS_FILE)
assert len(report["runs"]) == len(SEEDS) * len(SCHEDULES)
print("PASS: completed", len(report["runs"]), "verified runs of", len(SEEDS)*len(SCHEDULES))

staged_pair_focus        seed=42 step=   0 ID pair=0.500 OOD pair=0.500
staged_pair_focus        seed=42 step= 100 ID pair=0.500 OOD pair=0.500
staged_pair_focus        seed=42 step= 200 ID pair=0.504 OOD pair=0.500
staged_pair_focus        seed=42 step= 300 ID pair=0.492 OOD pair=0.490
staged_pair_focus        seed=42 step= 400 ID pair=0.504 OOD pair=0.482
staged_pair_focus        seed=42 step= 500 ID pair=0.500 OOD pair=0.492
staged_pair_focus        seed=42 step= 600 ID pair=0.500 OOD pair=0.490
staged_pair_focus        seed=42 step= 700 ID pair=0.504 OOD pair=0.492
staged_pair_focus        seed=42 step= 800 ID pair=0.504 OOD pair=0.490
staged_pair_focus        seed=42 step= 900 ID pair=0.504 OOD pair=0.500
staged_pair_focus        seed=42 step=1000 ID pair=0.500 OOD pair=0.495
staged_pair_focus        seed=42 step=1100 ID pair=0.500 OOD pair=0.497
staged_pair_focus        seed=42 step=1200 ID pair=0.500 OOD pair=0.500
staged_pair_focus        seed=42 step=1300 ID pair=0.500 OOD pai

## Verify the completed five-seed experiment

Expect **20 completed seed–schedule runs**, each with 9,000 updates and 91 evaluation checkpoints. All traces must match their corresponding seed, schedule, evaluation examples, and checkpoint steps.


In [ ]:
rows = []
for r in report["runs"]:
    if r["seed"] not in SEEDS:
        continue
    trace = validate_trace(OUT_DIR / r["prediction_trace"], r["seed"], r["schedule"])
    rows.append({"seed": r["seed"], "schedule": r["schedule"],
                 "steps": r["steps_run"],
                 "id_pair": r["final"]["id_dev"]["by_template"]["pair"],
                 "ood_pair": r["final"]["ood_dev"]["by_template"]["pair"],
                 "trace_steps": len(trace["splits"]["id_dev"]["steps"])})
display(pd.DataFrame(rows).sort_values(["seed","schedule"]))
assert len(rows) == len(SEEDS)*len(SCHEDULES)
print("PASS: all requested runs and traces are complete")

,seed,schedule,steps,id_pair,ood_pair,trace_steps
3,42,mixed,9000,0.656250,0.6125,91
2,42,mixed_pair_focus,9000,0.644531,0.6075,91
1,42,prereq_mixed_pair_focus,9000,0.644531,0.6000,91
0,42,staged_pair_focus,9000,0.652344,0.5575,91
7,43,mixed,9000,0.585938,0.5425,91
6,43,mixed_pair_focus,9000,0.605469,0.5550,91
5,43,prereq_mixed_pair_focus,9000,0.609375,0.5350,91
4,43,staged_pair_focus,9000,0.570312,0.5275,91
11,44,mixed,9000,0.582031,0.5450,91
10,44,mixed_pair_focus,9000,0.636719,0.6000,91


PASS: all requested runs and traces are complete
